# skill-call / tool-call flow

Verifies that `broflow` can support the two-phase design worked out in
discussion: `user-input -> skill-call -> tool-call (looped) -> fail-recovery -> answer`,
plus a shared `ask-follow-up-question` task either can drop into and resume from.

Key shape being tested, not just the individual tasks:

- **skill-call** runs once, picks zero or more skills via a scripted stand-in
  for `{"skill_names": [...]}`. `load_skill` itself is a free code step, not an
  LLM call.
- **tool-call** loops on *itself* -- `broflow.Flow.run` has no predecessor
  tracking, it just does `registry.get(current.next_action)` each step, so a
  task routing back to its own identifier works exactly like any other
  transition. Confirmed by reading `broflow/core.py` directly before building
  this.
- Picking `load_skill_extension` is handled as a free code step too: it
  extends `state.available_tools` (standing in for `ToolControl` discovering a
  skill's `scripts/*.py`), then loops back into tool-call rather than ending
  the turn.
- A failing tool routes to **fail-recovery**, which requeues a corrected call
  directly into `state.tool_calls` and sends control straight back to
  **tool-call** for execution -- not through skill-call or a fresh tool-call
  decision -- up to `MAX_RETRIES`, then falls through to **answer** with an
  honest failure message.
- **ask-follow-up-question** is one shared task, not one per caller. Whichever
  task decides it needs to ask (skill-call or tool-call) records itself in
  `state.return_to` before routing there; the task asks, collects the reply,
  and routes back to whatever `return_to` says -- so the same task serves both
  callers without needing to know who they are.

`skill_call_plan` / `tool_call_plan` are scripted stand-ins for the real
`skill-call`/`tool-call` LLM calls, now shaped as `{"skills"/"tool_use": [...],
"ask": str|None}` so a scripted decision can represent picking
`ask_followup_question` too -- `"ask"` holds the question text the model
would've written as its reply content alongside that tool call. `reply_plan`
is the matching stand-in for the user's scripted answer. Same pattern as
`Router.next_state` in `flow_idea.ipynb`: swap the pops for real model calls
later, the task contract (read state, call `set_next`) doesn't change.

In [ ]:
from dataclasses import dataclass, field
from enum import StrEnum
from typing import Any

from broflow import BaseTask, TaskRegistry, Flow


class Process(StrEnum):
    USER_INPUT = "user_input"
    SKILL_CALL = "skill_call"
    TOOL_CALL = "tool_call"
    ASK_FOLLOW_UP_QUESTION = "ask_follow_up_question"
    FAIL_RECOVERY = "fail_recovery"
    ANSWER = "answer"
    END = "end"


MAX_RETRIES = 3

# Toy skill -> tools map, standing in for what ToolControl would discover from
# a skill's scripts/ folder once load_skill_extension is asked to load one.
SKILL_TOOLS = {
    "read-file": ["read_file", "list_files"],
}

In [ ]:
@dataclass
class State:
    input: str = ''
    skill_names: list = field(default_factory=list)
    available_tools: list = field(default_factory=lambda: ["load_skill_extension", "ask_followup_question"])

    # Scripted stand-ins for what the real skill-call/tool-call LLM calls would
    # return -- popped one entry per visit, same "manual override" pattern used
    # for Router.next_state in flow_idea.ipynb. Each entry is
    # {"skills"/"tool_use": [...], "ask": str|None} -- "ask" set means the
    # model picked ask_followup_question, with this as its reply-content text.
    skill_call_plan: list = field(default_factory=list)
    tool_call_plan: list = field(default_factory=list)
    reply_plan: list = field(default_factory=list)  # scripted user replies

    return_to: Any = None
    follow_up_question: str = ''

    tool_calls: list = field(default_factory=list)
    tool_results: list = field(default_factory=list)
    error_message: str = ''
    retry_count: int = 0
    answer: str = ''

In [ ]:
class UserInput(BaseTask):
    possible_next = {Process.SKILL_CALL}

    def __call__(self, state: State):
        self.set_next(Process.SKILL_CALL)
        return state


class SkillCall(BaseTask):
    """Stand-in for the skill-call LLM call. Pops one scripted decision --
    real code would call the model and parse {"skill_names": [...]}."""
    possible_next = {Process.TOOL_CALL, Process.ASK_FOLLOW_UP_QUESTION, Process.ANSWER}

    def __call__(self, state: State):
        decision = state.skill_call_plan.pop(0) if state.skill_call_plan else {"skills": [], "ask": None}

        if decision.get("ask"):
            state.follow_up_question = decision["ask"]
            state.return_to = Process.SKILL_CALL
            self.set_next(Process.ASK_FOLLOW_UP_QUESTION)
            return state

        picked = decision.get("skills", [])
        state.skill_names = picked

        if not picked:
            state.answer = "I couldn't find a skill that matches this request."
            self.set_next(Process.ANSWER)
            return state

        # load_skill: code only, no LLM call -- registers this skill's own
        # extension-gated tools (still just load_skill_extension at first).
        self.set_next(Process.TOOL_CALL)
        return state

In [ ]:
class ToolCall(BaseTask):
    """Stand-in for the tool-call LLM call, looped. If state.tool_calls is
    already populated (FailRecovery queued a retry), run that directly instead
    of asking for a fresh decision -- mirrors ToolUse's original design in
    flow_idea.ipynb."""
    possible_next = {Process.TOOL_CALL, Process.ASK_FOLLOW_UP_QUESTION, Process.FAIL_RECOVERY, Process.ANSWER}

    def __call__(self, state: State):
        if not state.tool_calls:
            decision = state.tool_call_plan.pop(0) if state.tool_call_plan else {"tool_use": [], "ask": None}

            if decision.get("ask"):
                state.follow_up_question = decision["ask"]
                state.return_to = Process.TOOL_CALL
                self.set_next(Process.ASK_FOLLOW_UP_QUESTION)
                return state

            state.tool_calls = decision.get("tool_use", [])

        if not state.tool_calls:
            self.set_next(Process.ANSWER)
            return state

        any_failed = False
        for call in state.tool_calls:
            name = call["name"]

            if name == "load_skill_extension":
                skill_name = call["input"]["skill_name"]
                path = call["input"]["path"]
                new_tools = [t for t in SKILL_TOOLS.get(skill_name, []) if t not in state.available_tools]
                state.available_tools.extend(new_tools)
                state.tool_results.append({
                    "name": name, "input": call["input"],
                    "output": f"loaded {path}; available_tools now includes {new_tools}",
                    "success": True,
                })
                continue

            failed = bool(call.get("input", {}).get("force_fail"))
            state.tool_results.append({
                "name": name,
                "input": call["input"],
                "output": f"tool '{name}' failed: simulated failure" if failed else f"mock result from {name}",
                "success": not failed,
            })
            any_failed = any_failed or failed

        state.tool_calls = []

        if any_failed and state.retry_count < MAX_RETRIES:
            state.error_message = next(r["output"] for r in state.tool_results if not r["success"])
            self.set_next(Process.FAIL_RECOVERY)
        else:
            self.set_next(Process.TOOL_CALL)
        return state


class AskFollowUpQuestion(BaseTask):
    """Asks + collects the reply in one step -- input() already blocks, so
    there's no separate pause/resume boundary to model as two tasks. Routes
    back to whichever task asked (state.return_to), not a fixed destination --
    this is what lets skill-call and tool-call share one task instead of each
    needing its own."""
    possible_next = {Process.SKILL_CALL, Process.TOOL_CALL}

    def __call__(self, state: State):
        reply = state.reply_plan.pop(0) if state.reply_plan else input(state.follow_up_question)
        state.input = reply
        state.follow_up_question = ''
        self.set_next(state.return_to)
        return state

In [ ]:
class FailRecovery(BaseTask):
    """Narrower than tool-call: only ever reasons about the one call that just
    failed, so it requeues directly into tool_calls and goes straight back to
    TOOL_CALL for execution -- not through another tool-call decision."""
    possible_next = {Process.TOOL_CALL, Process.ANSWER}

    def __call__(self, state: State):
        state.retry_count += 1
        if state.retry_count >= MAX_RETRIES:
            state.answer = f"I couldn't complete this after {state.retry_count} attempts: {state.error_message}"
            self.set_next(Process.ANSWER)
            return state

        last_failed = next(r for r in reversed(state.tool_results) if not r["success"])
        corrected_input = {k: v for k, v in last_failed["input"].items() if k != "force_fail"}
        state.tool_calls.append({"name": last_failed["name"], "input": corrected_input})
        self.set_next(Process.TOOL_CALL)
        return state


class Answer(BaseTask):
    possible_next = {Process.END}

    def __call__(self, state: State):
        if not state.answer:
            state.answer = "Here's what I found."
        self.set_next(Process.END)
        return state

In [ ]:
registry = TaskRegistry()
registry.register(Process.USER_INPUT, UserInput(Process.USER_INPUT.value))
registry.register(Process.SKILL_CALL, SkillCall(Process.SKILL_CALL.value))
registry.register(Process.TOOL_CALL, ToolCall(Process.TOOL_CALL.value))
registry.register(Process.ASK_FOLLOW_UP_QUESTION, AskFollowUpQuestion(Process.ASK_FOLLOW_UP_QUESTION.value))
registry.register(Process.FAIL_RECOVERY, FailRecovery(Process.FAIL_RECOVERY.value))
registry.register(Process.ANSWER, Answer(Process.ANSWER.value))

flow = Flow(registry)

## Demo A -- skill found, extension-gated tool, one failure, recovers, succeeds

`skill-call` picks `read-file`. First `tool-call` visit asks for
`load_skill_extension` (loading `scripts/read_file.py`), which extends
`available_tools` and loops back into `tool-call` rather than ending the turn.
Second visit calls the now-available `read_file` tool with `force_fail=True`,
which routes to `fail-recovery`; it requeues a corrected call straight back
into `tool-call`, which succeeds, loops once more, finds nothing left queued,
and falls through to `answer`.

In [ ]:
state = State(
    input="what's in skills/read-file/SKILL.md?",
    skill_call_plan=[{"skills": ["read-file"], "ask": None}],
    tool_call_plan=[
        {"tool_use": [{"name": "load_skill_extension", "input": {"skill_name": "read-file", "path": "scripts/read_file.py"}}], "ask": None},
        {"tool_use": [{"name": "read_file", "input": {"pattern": "skills/read-file/SKILL.md", "force_fail": True}}], "ask": None},
    ],
)
flow.run(start=Process.SKILL_CALL, end=Process.END, state=state)
print("A trace:", flow.trace)
print("A available_tools:", state.available_tools)
print("A answer:", state.answer)
assert [t for t, _ in flow.trace] == [
    "skill_call", "tool_call", "tool_call", "fail_recovery", "tool_call", "tool_call", "answer",
]
assert "read_file" in state.available_tools and "list_files" in state.available_tools
assert state.retry_count == 1

## Demo B -- no skill matches, straight to answer

`skill-call` returns an empty `skill_names` -- the flow never touches
`tool-call` at all.

In [ ]:
state = State(input="tell me a joke about pointers", skill_call_plan=[{"skills": [], "ask": None}])
flow.run(start=Process.SKILL_CALL, end=Process.END, state=state)
print("B trace:", flow.trace)
print("B answer:", state.answer)
assert [t for t, _ in flow.trace] == ["skill_call", "answer"]
assert "couldn't find a skill" in state.answer

## Demo C -- skill-call asks a follow-up, resumes itself

First `skill-call` visit has nothing to pick and asks a clarifying question
instead, setting `return_to = SKILL_CALL`. `ask-follow-up-question` collects
the (scripted) reply and routes straight back to `skill-call`, which now picks
`read-file` on its second visit and proceeds into `tool-call` as normal.

In [ ]:
state = State(
    skill_call_plan=[
        {"skills": [], "ask": "Which topic are you asking about?"},
        {"skills": ["read-file"], "ask": None},
    ],
    tool_call_plan=[{"tool_use": [], "ask": None}],
    reply_plan=["files in the project"],
)
flow.run(start=Process.SKILL_CALL, end=Process.END, state=state)
print("C trace:", flow.trace)
print("C input (last reply):", state.input)
print("C skill_names:", state.skill_names)
print("C answer:", state.answer)
assert [t for t, _ in flow.trace] == [
    "skill_call", "ask_follow_up_question", "skill_call", "tool_call", "answer",
]
assert state.input == "files in the project"
assert state.skill_names == ["read-file"]

## Demo D -- tool-call asks a follow-up, resumes itself

`skill-call` finds `read-file` immediately. First `tool-call` visit has
nothing concrete to run and asks which file was meant, setting
`return_to = TOOL_CALL` this time -- the *same* `ask-follow-up-question` task
as Demo C, just routed back to a different caller. It resumes tool-call, which
now runs `read_file` for real, loops once more to confirm nothing's left
queued, and falls through to `answer`.

In [ ]:
state = State(
    skill_call_plan=[{"skills": ["read-file"], "ask": None}],
    tool_call_plan=[
        {"tool_use": [], "ask": "Which file did you mean?"},
        {"tool_use": [{"name": "read_file", "input": {"pattern": "skills/read-file/SKILL.md"}}], "ask": None},
        {"tool_use": [], "ask": None},
    ],
    reply_plan=["skills/read-file/SKILL.md"],
)
flow.run(start=Process.SKILL_CALL, end=Process.END, state=state)
print("D trace:", flow.trace)
print("D input (last reply):", state.input)
print("D tool_results:", state.tool_results)
print("D answer:", state.answer)
assert [t for t, _ in flow.trace] == [
    "skill_call", "tool_call", "ask_follow_up_question", "tool_call", "tool_call", "answer",
]
assert state.input == "skills/read-file/SKILL.md"
assert state.tool_results[0]["success"] is True